# L10d: Face Arithmetic with Autoencoders
In this lab, we train a nonlinear autoencoder on the [Olivetti Faces dataset](https://scikit-learn.org/stable/datasets/real_world.html#olivetti-faces-dataset), a collection of 400 grayscale face images (40 subjects, 10 images each, each $64\times 64$ pixels). After training, we explore the learned latent space by interpolating between faces and performing latent space arithmetic, the face equivalent of the word vector analogies from L9a.

The idea is simple: if the autoencoder learns a structured latent space, then vector operations on the codes $\mathbf{z}$ should produce meaningful changes in the reconstructed faces. For example, if we compute the "glasses vector" by subtracting the code for a face without glasses from the code for the same face with glasses, we can add that vector to a new face to synthesize what that person might look like wearing glasses.

## Setup, Data, and Prerequisites
We set up the computational environment, load the Olivetti Faces dataset, build and train the autoencoder, and compute all the representations we need for the tasks below.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file in the current module's global scope. The `Include.jl` file loads required packages and defines path constants.

In [ ]:
include("Include.jl"); # load packages, paths, and local source files

### Constants

In [ ]:
d = 32;                 # bottleneck dimension
image_dim = 64;         # each face is 64x64 pixels
input_dim = 64 * 64;    # flattened image dimension (4096)
n_subjects = 40;        # number of subjects in the dataset
n_per_subject = 10;     # images per subject
η = 1e-3;               # learning rate
num_epochs = 200;       # training epochs
batch_size = 32;        # mini-batch size

### Data
The [Olivetti Faces dataset](https://scikit-learn.org/stable/datasets/real_world.html#olivetti-faces-dataset) contains 400 grayscale images of 40 subjects (10 images per subject). Each image is $64\times 64$ pixels with values in $[0,1]$. The images vary in lighting, facial expression, and whether the subject is wearing glasses.

In [ ]:
X, y = let

    # load the face data and targets from CSV -
    faces_df = CSV.read(joinpath(_PATH_TO_DATA, "olivetti_faces.csv"), DataFrame; header=false);
    targets_df = CSV.read(joinpath(_PATH_TO_DATA, "olivetti_targets.csv"), DataFrame; header=false);

    # convert to matrices: X is (4096 x 400), each column is one face -
    X = Float32.(Matrix(faces_df)');
    y = Int.(vec(Matrix(targets_df)));

    println("Face matrix size: $(size(X)) (pixels x images)");
    println("Subjects: $(length(unique(y))) with $(n_per_subject) images each");

    (X, y)
end;

Let's visualize the first image of each of the 40 subjects.

In [ ]:
let
    plots = [];
    for subject in 0:39
        idx = findfirst(y .== subject);
        img = Gray.(reshape(X[:, idx], image_dim, image_dim)');
        push!(plots, heatmap(img, color=:grays, axis=false, ticks=false, 
            title="S$(subject)", titlefontsize=7, aspect_ratio=:equal));
    end
    plot(plots..., layout=(5, 8), size=(900, 600), margin=0Plots.mm)
end

### Build and Train the Autoencoder
We build a nonlinear autoencoder with a symmetric architecture. The encoder compresses each $4096$-dimensional face into a $d = 32$-dimensional bottleneck code, and the decoder mirrors the encoder to reconstruct the face from that code.

> __Architecture:__ The encoder has layers $4096 \rightarrow 512 \rightarrow 128 \rightarrow 32$ and the decoder mirrors this as $32 \rightarrow 128 \rightarrow 512 \rightarrow 4096$. We use `relu` activations in hidden layers and `sigmoid` on the decoder output to keep pixel values in $[0,1]$. The model is trained by minimizing the MSE reconstruction loss $\mathcal{L}(\theta) = \frac{1}{N}\sum_{n=1}^{N}\|\mathbf{x}^{(n)} - \hat{\mathbf{x}}^{(n)}(\theta)\|_{2}^{2}$ using the Adam optimizer.

In [ ]:
encoder = Chain(
    Dense(input_dim, 512, relu),
    Dense(512, 128, relu),
    Dense(128, d)
);

decoder = Chain(
    Dense(d, 128, relu),
    Dense(128, 512, relu),
    Dense(512, input_dim, sigmoid)
);

autoencoder = Chain(encoder, decoder);
n_params = sum(length, Flux.params(autoencoder));
println("Encoder: $(input_dim) → 512 → 128 → $(d)");
println("Decoder: $(d) → 128 → 512 → $(input_dim)");
println("Total parameters: $(n_params)")

In [ ]:
loss_history = let

    # setup -
    opt_state = Flux.setup(Adam(η), autoencoder);
    history = Vector{Float64}();
    N = size(X, 2);

    # training loop -
    for epoch in 1:num_epochs
        epoch_loss = 0.0;
        indices = shuffle(1:N);
        n_batches = 0;

        for start in 1:batch_size:N
            stop = min(start + batch_size - 1, N);
            batch = X[:, indices[start:stop]];

            loss, grads = Flux.withgradient(autoencoder) do model
                x_hat = model(batch);
                Flux.mse(x_hat, batch);
            end

            Flux.update!(opt_state, autoencoder, grads[1]);
            epoch_loss += loss;
            n_batches += 1;
        end

        avg_loss = epoch_loss / n_batches;
        push!(history, avg_loss);

        if epoch % 25 == 0 || epoch == 1
            println("Epoch $(lpad(epoch, 3)): loss = $(round(avg_loss, digits=6))");
        end
    end

    history
end;

Let's plot the training loss to verify convergence.

In [ ]:
let
    plot(1:num_epochs, loss_history;
        xlabel="Epoch", ylabel="Average MSE Loss",
        title="Face Autoencoder Training Loss",
        legend=false, lw=2)
end

### Precompute Representations
We compute the mean latent code for each subject (averaging over all 10 images) and the PCA projection for comparison in the tasks below.

In [ ]:
# mean autoencoder codes per subject -
Z_mean = let
    Z = zeros(Float32, d, n_subjects);
    for subject in 0:(n_subjects - 1)
        indices = findall(y .== subject);
        Z_subject = encoder(X[:, indices]);
        Z[:, subject + 1] = vec(mean(Z_subject, dims=2));
    end
    Z
end;

# PCA projection for comparison -
V_d, μ_pca, Z_pca_mean = let

    X_f64 = Float64.(X);
    μ = mean(X_f64, dims=2);
    X_centered = X_f64 .- μ;

    U, S, V = svd(X_centered');
    V_d = V[:, 1:d];
    Z_pca = V_d' * X_centered;

    Z_pca_mean = zeros(Float64, d, n_subjects);
    for subject in 0:(n_subjects - 1)
        indices = findall(y .== subject);
        Z_pca_mean[:, subject + 1] = vec(mean(Z_pca[:, indices], dims=2));
    end

    println("Variance explained by top $(d) PCs: $(round(100 * sum(S[1:d].^2) / sum(S.^2), digits=1))%");

    (V_d, μ, Z_pca_mean)
end;

___

## Task 1: Reconstructions and Face Morphing
We start by checking that the autoencoder learned useful representations, then explore the latent space by interpolating between two faces.

> __Reconstructions:__ If the autoencoder is working, the reconstructed faces should preserve identity even though they are blurrier than the originals. The blurriness comes from the MSE loss, which penalizes large pixel errors but not blurriness directly.
>
> __Interpolation:__ If the latent space is smooth, a linear path between two codes $\mathbf{z}(\alpha) = (1 - \alpha)\mathbf{z}_A + \alpha\mathbf{z}_B$ should produce a gradual morph between two faces, not abrupt jumps.

In [ ]:
let
    subjects_to_show = [0, 5, 10, 15, 20, 25, 30, 35];
    n = length(subjects_to_show);
    plots = [];

    for subject in subjects_to_show
        idx = findfirst(y .== subject);

        img_orig = Gray.(reshape(X[:, idx], image_dim, image_dim)');
        push!(plots, heatmap(img_orig, color=:grays, axis=false, ticks=false,
            title="S$(subject) orig", titlefontsize=7, aspect_ratio=:equal));

        x_hat = autoencoder(X[:, idx]);
        img_recon = Gray.(reshape(x_hat, image_dim, image_dim)');
        push!(plots, heatmap(img_recon, color=:grays, axis=false, ticks=false,
            title="S$(subject) recon", titlefontsize=7, aspect_ratio=:equal));
    end

    plot(plots..., layout=(n, 2), size=(300, 100 * n), margin=0Plots.mm)
end

Now let's morph between two subjects. Try changing `subject_A` and `subject_B` to explore different pairs.

In [ ]:
let
    subject_A = 0;
    subject_B = 10;
    idx_A = findfirst(y .== subject_A);
    idx_B = findfirst(y .== subject_B);

    z_A = encoder(X[:, idx_A]);
    z_B = encoder(X[:, idx_B]);

    n_steps = 10;
    alphas = range(0.0, 1.0, length=n_steps);
    plots = [];

    for α in alphas
        z_interp = (1 - α) .* z_A .+ α .* z_B;
        x_hat = decoder(z_interp);
        img = Gray.(reshape(x_hat, image_dim, image_dim)');
        push!(plots, heatmap(img, color=:grays, axis=false, ticks=false,
            title="α=$(round(α, digits=1))", titlefontsize=7, aspect_ratio=:equal));
    end

    plot(plots..., layout=(1, n_steps), size=(900, 120), margin=0Plots.mm)
end

### Things to think about
* __Question:__ We used a bottleneck dimension of $d = 32$ for a compression ratio of $4096 / 32 = 128\times$. What would happen if you decreased $d$ to $8$? What about increasing it to $256$?
* __Question:__ What would happen if we interpolated in pixel space instead of latent space (i.e., $(1-\alpha)\mathbf{x}_A + \alpha\mathbf{x}_B$ directly)? Would the intermediate images look like faces?

___

## Task 2: Face Arithmetic in Latent Space
In L9a, we saw that word vectors support analogies like $\mathbf{v}_{\text{king}} - \mathbf{v}_{\text{man}} + \mathbf{v}_{\text{woman}} \approx \mathbf{v}_{\text{queen}}$. We can try the same thing with faces. The idea is to isolate an attribute (like smiling or wearing glasses) as a direction in latent space, then apply that direction to a new face.

> __How face arithmetic works:__ Suppose subject $A$ has two images: one smiling ($\mathbf{x}_{A,\text{smile}}$) and one neutral ($\mathbf{x}_{A,\text{neutral}}$). The "smile vector" is $\Delta\mathbf{z}_{\text{smile}} = f_{\mathrm{enc}}(\mathbf{x}_{A,\text{smile}}) - f_{\mathrm{enc}}(\mathbf{x}_{A,\text{neutral}})$. To add a smile to subject $B$, we decode $f_{\mathrm{enc}}(\mathbf{x}_{B}) + \beta\cdot\Delta\mathbf{z}_{\text{smile}}$ where $\beta$ controls the strength.

First, let's browse the images for several subjects to find useful attribute pairs (e.g., neutral vs. smiling, no glasses vs. glasses).

In [ ]:
let
    subjects_to_browse = [0, 2, 10, 12, 33, 36];
    
    for subject in subjects_to_browse
        indices = findall(y .== subject);
        plots = [];
        for (img_num, idx) in enumerate(indices)
            img = Gray.(reshape(X[:, idx], image_dim, image_dim)');
            push!(plots, heatmap(img, color=:grays, axis=false, ticks=false,
                title="img $(img_num)", titlefontsize=7, aspect_ratio=:equal));
        end
        p = plot(plots..., layout=(1, length(indices)), size=(900, 120), margin=0Plots.mm,
            plot_title="Subject $(subject)", plot_titlefontsize=9);
        display(p)
    end
end

Now extract an attribute vector from one subject and apply it to another with varying strength $\beta$. Edit the subject and image indices to try different attribute transfers.

In [ ]:
let
    # --- Step 1: Define the attribute vector ---
    source_subject = 10;
    img_with_attribute = 1;     # e.g., smiling (1-indexed within subject)
    img_without_attribute = 4;  # e.g., neutral

    source_indices = findall(y .== source_subject);
    z_with = encoder(X[:, source_indices[img_with_attribute]]);
    z_without = encoder(X[:, source_indices[img_without_attribute]]);
    Δz = z_with .- z_without;

    # --- Step 2: Apply to a target subject ---
    target_subject = 0;
    target_img = 1;
    target_indices = findall(y .== target_subject);
    z_target = encoder(X[:, target_indices[target_img]]);

    # --- Step 3: Decode with varying strength ---
    strengths = [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0];
    plots = [];

    for β in strengths
        z_modified = z_target .+ β .* Δz;
        x_hat = decoder(z_modified);
        img = Gray.(reshape(x_hat, image_dim, image_dim)');
        push!(plots, heatmap(img, color=:grays, axis=false, ticks=false,
            title="β=$(β)", titlefontsize=7, aspect_ratio=:equal));
    end

    plot(plots..., layout=(1, length(strengths)), size=(900, 140), margin=0Plots.mm,
        plot_title="S$(source_subject) attribute → S$(target_subject)", plot_titlefontsize=9)
end

We can also do word2vec-style analogies using the mean codes: $\bar{\mathbf{z}}_A - \bar{\mathbf{z}}_B + \bar{\mathbf{z}}_C = ?$

In [ ]:
let
    # face analogy: A - B + C = ?
    A = 33; B = 0; C = 12;

    z_result = Z_mean[:, A + 1] .- Z_mean[:, B + 1] .+ Z_mean[:, C + 1];

    img_A = Gray.(reshape(decoder(Z_mean[:, A + 1]), image_dim, image_dim)');
    img_B = Gray.(reshape(decoder(Z_mean[:, B + 1]), image_dim, image_dim)');
    img_C = Gray.(reshape(decoder(Z_mean[:, C + 1]), image_dim, image_dim)');
    img_result = Gray.(reshape(decoder(z_result), image_dim, image_dim)');

    p1 = heatmap(img_A, color=:grays, axis=false, ticks=false,
        title="S$(A)", titlefontsize=9, aspect_ratio=:equal);
    p2 = heatmap(img_B, color=:grays, axis=false, ticks=false,
        title="- S$(B)", titlefontsize=9, aspect_ratio=:equal);
    p3 = heatmap(img_C, color=:grays, axis=false, ticks=false,
        title="+ S$(C)", titlefontsize=9, aspect_ratio=:equal);
    p4 = heatmap(img_result, color=:grays, axis=false, ticks=false,
        title="= ?", titlefontsize=9, aspect_ratio=:equal);

    plot(p1, p2, p3, p4, layout=(1, 4), size=(600, 180), margin=0Plots.mm)
end

### Things to think about
* __Question:__ The face arithmetic works better for some attributes than others. Does a standard autoencoder have any explicit mechanism to disentangle facial attributes? What might you add to the training objective to encourage disentanglement?
* __Experiment:__ Try several different $(A, B, C)$ triples. Can you find a combination where the result clearly shows a transferred attribute (e.g., glasses, head tilt, expression)?

___

## Task 3: Compare to PCA
In the lecture (L10c), we showed that a linear autoencoder recovers the PCA subspace. The PCA projection computed in the setup section is essentially what a linear autoencoder would produce. Let's repeat the face analogy from Task 2 using PCA codes and compare the results side by side.

> __What to look for:__ PCA arithmetic operates in a linear subspace while the autoencoder operates in a nonlinear latent space. The autoencoder result should look more like a plausible face because the decoder maps codes back onto the "face manifold," while PCA reconstruction can produce pixel values that do not look face-like.

In [ ]:
let
    A = 33; B = 0; C = 12;

    # --- Autoencoder arithmetic ---
    z_ae_result = Z_mean[:, A + 1] .- Z_mean[:, B + 1] .+ Z_mean[:, C + 1];
    img_ae = Gray.(reshape(decoder(z_ae_result), image_dim, image_dim)');

    # --- PCA arithmetic ---
    z_pca_result = Z_pca_mean[:, A + 1] .- Z_pca_mean[:, B + 1] .+ Z_pca_mean[:, C + 1];
    x_pca_recon = V_d * z_pca_result .+ μ_pca;
    x_pca_recon = clamp.(x_pca_recon, 0.0, 1.0);
    img_pca = Gray.(reshape(x_pca_recon, image_dim, image_dim)');

    # --- Original faces for reference ---
    img_A = Gray.(reshape(decoder(Z_mean[:, A + 1]), image_dim, image_dim)');
    img_B = Gray.(reshape(decoder(Z_mean[:, B + 1]), image_dim, image_dim)');
    img_C = Gray.(reshape(decoder(Z_mean[:, C + 1]), image_dim, image_dim)');

    p1 = heatmap(img_A, color=:grays, axis=false, ticks=false,
        title="S$(A)", titlefontsize=8, aspect_ratio=:equal);
    p2 = heatmap(img_B, color=:grays, axis=false, ticks=false,
        title="- S$(B)", titlefontsize=8, aspect_ratio=:equal);
    p3 = heatmap(img_C, color=:grays, axis=false, ticks=false,
        title="+ S$(C)", titlefontsize=8, aspect_ratio=:equal);
    p4 = heatmap(img_ae, color=:grays, axis=false, ticks=false,
        title="AE result", titlefontsize=8, aspect_ratio=:equal);
    p5 = heatmap(img_pca, color=:grays, axis=false, ticks=false,
        title="PCA result", titlefontsize=8, aspect_ratio=:equal);

    plot(p1, p2, p3, p4, p5, layout=(1, 5), size=(750, 180), margin=0Plots.mm)
end

### Things to think about
* __Question:__ Which method produces more recognizable faces in the analogy result? What does this tell you about the geometry of face space?
* __Question:__ The PCA result is what a linear autoencoder would produce. How does this comparison illustrate why nonlinear activations matter?

___

## Summary
In this lab, we trained a nonlinear autoencoder on the Olivetti Faces dataset and explored three operations in the learned latent space: reconstruction, interpolation (face morphing), and arithmetic (face analogies). We compared the autoencoder's latent space to PCA to illustrate the value of nonlinear representations. The key takeaway is that the same latent space arithmetic that makes word2vec analogies work (L9a) also applies to image embeddings learned by autoencoders.